# このノートだけ

開くのはこのページだけです。ほかのノートは開かない。

動画は焼きません。投稿しません。

## 押す順番

1. **読み込み**
2. **選ぶ** で、やりたいことを1つ選んで実行
3. **実行**

見た目を変えるときだけ、2と3のあいだに、話の名前が同じ見た目のセルを1つ実行します。変えないときは飛ばします。初期値です。

上のメニューの「すべてのセルを実行」は押しません。

## 選び方

**表を見る**

美容、ドッグフード、見守りカメラ、婚活の作り方の表が出ます。ジョブは書きません。

**話でジョブを書く**

話を1つ選びます。ドッグフードだけ、インタビュー・咀嚼・ダンス・会話も選びます。

中身を1つ選びます。テーマは書きません。台詞は入っています。

- **テンプレ** … 用意してある短い場面と台詞
- **オマージュ** … 同じ絵とカメラの順で、別の短い台詞
- **元の型のまま** … 型の絵とカメラの順。短い新しい台詞。元の顔、元の台詞、曲名は入りません

静止画に、画像ファイルの場所を書きます。台詞を変えたいときだけ、台詞の欄に1行ずつ書きます。

**自分の文で1本（手入力で I2V）**

静止画とプロンプトを自分で書きます。秒と画面も「選ぶ」で選びます。「この場で焼く」を入れたときだけ、このランタイムで1本焼きます。重みが要ります。入れなければコマンドを書くだけです。

## 4つの話

1. **美容：材料のキャラ**（the.care.logic）
   材料のキャラが1体で悩み、対処して、笑顔で終わる。30秒を10秒ずつ3つ。見た目を変えるときは材料・場所・口調。

2. **ドッグフード：犬**（nuts0629）
   同じ犬。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。見た目を変えるときは犬・場所・口調。型を選ぶのはこの話だけ。

3. **見守り：猫2匹の言い合い**（junjun_ranran）
   猫が2匹で言い合い、最後に人が少し出る。45秒。見守りカメラの映像ではない。見た目を変えるときは猫2匹・人・場所・口調。

4. **婚活：男女の会話**（yako.shiawasekon）
   成人の男女が会話して、最後に関係が変わる。60秒。見た目を変えるときは女性・男性・場所・口調。

一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や、元のアカウントの人・動物に似せる文は、そこで止まります。

秒数・カット・字幕は型のままです。

Checkpoint は MiniMax-H3。LoRA は FL2VA の Turbo と、動作のつながり。同じものを、話のジョブと手入力の I2V で使います。


In [ ]:
#@title 読み込み { display-mode: "form" }
#@markdown 最初にこのセルを実行する。調査日は 2026-10-03。初めてなら変えない。
DATE = "2026-10-03"  #@param {type:"string"}

from pathlib import Path
import sys
import urllib.request

BRANCH = "cursor/affi-template-bake-44d6"
REPO = "fireworker011/Research"
RAW = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/research/affi-templates"
STRONG_MIN = 10
COMPARE_MIN = 3
THIN_BELOW = 10
MIN_GAP_PT = 10
LOCAL = Path("research/affi-templates")
if not (LOCAL / "affi_bake.py").is_file():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "research/affi-templates/affi_bake.py").is_file():
            LOCAL = candidate / "research/affi-templates"
            break
if not (LOCAL / "affi_bake.py").is_file():
    LOCAL = Path("/content/affi-templates") if Path("/content").is_dir() else Path(".affi-templates-download")
    files = [
        "affi_genre_templates.py",
        "affi_reference.py",
        "affi_av.py",
        "affi_bake.py",
        "reference-accounts/hypotheses.yaml",
        "reference-accounts/results.csv",
        "reference-accounts/looks.yaml",
        "reference-accounts/templates/the.care.logic.yaml",
        "reference-accounts/templates/nuts0629.yaml",
        "reference-accounts/templates/junjun_ranran.yaml",
        "reference-accounts/templates/yako.shiawasekon.yaml",
        f"data/{DATE}/accounts.csv",
    ]
    for rel in files:
        dest = LOCAL / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{rel}", dest)
        print("取りました", rel)
    try:
        urllib.request.urlretrieve(f"{RAW}/data/{DATE}/videos.csv", LOCAL / "data" / DATE / "videos.csv")
    except Exception:
        print("動画一覧は無し。アカウント一覧だけで計算する。")
sys.path.insert(0, str(LOCAL))
import affi_reference as ref
ref.ROOT = LOCAL
ref.TEMPLATES = LOCAL / "reference-accounts" / "templates"
ref.HYPOTHESES = LOCAL / "reference-accounts" / "hypotheses.yaml"
ref.RESULTS = LOCAL / "reference-accounts" / "results.csv"
ref.LOOKS = LOCAL / "reference-accounts" / "looks.yaml"
DATA_DIR = LOCAL / "data" / DATE
TABLE_OUT = LOCAL / "templates"
print("読みました", LOCAL)
print("次は「選ぶ」を実行してください。")


In [ ]:
#@title 選ぶ { display-mode: "form" }
#@markdown やりたいことを1つ選んで、このセルを実行する。上の「すべてのセルを実行」は押さない。
何をする = "話でジョブを書く" #@param ["表を見る", "話でジョブを書く", "自分の文で1本"]
#@markdown 話でジョブを書くときだけ、下の話・型・中身・台詞を使う。表を見る、自分の文では無視する。
話 = "ドッグフード：犬（nuts0629）" #@param ["美容：材料のキャラ（the.care.logic）", "ドッグフード：犬（nuts0629）", "見守り：猫2匹の言い合い（junjun_ranran）", "婚活：男女の会話（yako.shiawasekon）"]
#@markdown ドッグフードだけ下を使う。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。ほかの話では無視する。
ドッグフードの型 = "インタビュー（既定）" #@param ["インタビュー（既定）", "咀嚼", "ダンス", "会話"]
#@markdown 中身。テンプレは用意した短い場面。オマージュは同じ絵の順で、別の短い台詞。元の型のままは型の絵とカメラ。元の顔、元の台詞、曲名は入らない。
中身 = "テンプレ" #@param ["テンプレ", "オマージュ", "元の型のまま"]
#@markdown 台詞を置き換えるときだけ書く。空なら、中身の台詞を使う。1行が1カット。
台詞 = "" #@param {type:"raw"}
#@markdown 静止画は画像ファイルの場所。話でジョブを書くときと、自分の文で1本のときに使う。
静止画 = "" #@param {type:"string"}
#@markdown 自分の文で1本（手入力で I2V）のときだけ、下を使う。
秒 = "10" #@param ["5", "6", "8", "10", "12", "14"]
画面 = "9:16" #@param ["9:16", "16:9"]
プロンプト = "" #@param {type:"raw"}
この場で焼く = False #@param {type:"boolean"}

import affi_bake

HANDLE, MODE = affi_bake.resolve_account(話, ドッグフードの型)
print(affi_bake.next_step(何をする))
if 何をする == "話でジョブを書く":
    print(affi_bake.describe_account(HANDLE, MODE))


# 見た目を変えるときだけ

「選ぶ」が **話でジョブを書く** のときだけ使います。見出しが今の話と同じセルを1つ実行します。

迷ったら、この4つは実行しません。初期値を使います。初期値は元のアカウントの顔ではありません。

ほかの話のセルを実行しても、見た目は入りません。


In [ ]:
#@title 美容：材料のキャラの見た目 { display-mode: "form" }
#@markdown 材料のキャラが1体で悩み、対処して、笑顔で終わる。30秒を10秒ずつ3つ。迷ったら初期値のまま。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 材料のキャラ
材料 = "茶葉" #@param ["茶葉", "石けん", "タオル", "マグカップ", "その他（直接入力）"] {allow-input: true}
作り = "粘土" #@param ["粘土", "折り紙", "水彩", "シンプルな3D", "その他（直接入力）"] {allow-input: true}
キャラの色 = "生成り" #@param ["生成り", "ミント", "紺", "黄土", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "the.care.logic")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "材料": 材料,
            "作り": 作り,
            "キャラの色": キャラの色,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title ドッグフード：犬の見た目 { display-mode: "form" }
#@markdown 同じ犬。インタビューは人がマイクを向ける8秒。咀嚼は8秒を4回。ダンスは全身で10秒。会話は吹き出し。人の顔はここでは選びません。迷ったら初期値のまま。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 犬（会話の2匹もこの見た目）
犬の種類 = "犬" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
犬種 = "柴" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
犬の毛 = "黒の短毛" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
犬の体 = "小柄" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
犬の衣装 = "無地の首輪" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "nuts0629")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "犬の種類": 犬の種類,
            "犬種": 犬種,
            "犬の毛": 犬の毛,
            "犬の体": 犬の体,
            "犬の衣装": 犬の衣装,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title 見守り：猫2匹の見た目 { display-mode: "form" }
#@markdown 猫が2匹で言い合い、最後に人が少し出る。45秒。見守りカメラの映像ではない。迷ったら初期値のまま（猫が2匹）。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 1匹目の猫
一匹目の種類 = "猫" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
一匹目の猫種 = "マンチカン" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
一匹目の毛 = "グレー一色" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
一匹目の体 = "普通" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
一匹目の衣装 = "何も着けない" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 2匹目の猫
二匹目の種類 = "猫" #@param ["犬", "猫", "うさぎ", "ハムスター", "鳥", "その他（直接入力）"] {allow-input: true}
二匹目の猫種 = "スコティッシュフォールド" #@param ["柴", "プードル", "ビーグル", "マンチカン", "スコティッシュフォールド", "その他（直接入力）"] {allow-input: true}
二匹目の毛 = "白の短毛" #@param ["黒の短毛", "白の短毛", "茶色のぶち", "グレー一色", "クリームの短毛", "その他（直接入力）"] {allow-input: true}
二匹目の体 = "小柄" #@param ["小柄", "普通", "がっしり", "脚が長め", "その他（直接入力）"] {allow-input: true}
二匹目の衣装 = "何も着けない" #@param ["無地の首輪", "ニット", "何も着けない", "バンダナ", "その他（直接入力）"] {allow-input: true}

#@markdown 最後に少し出る人
人の性別 = "女性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
人の年代 = "40代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
人の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
人の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
人の服 = "無地のニット" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
人の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
人の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "リビング" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "北欧" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "白と木" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "昼の自然光" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調（2匹共通）
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "junjun_ranran")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "一匹目の種類": 一匹目の種類,
            "一匹目の猫種": 一匹目の猫種,
            "一匹目の毛": 一匹目の毛,
            "一匹目の体": 一匹目の体,
            "一匹目の衣装": 一匹目の衣装,
            "二匹目の種類": 二匹目の種類,
            "二匹目の猫種": 二匹目の猫種,
            "二匹目の毛": 二匹目の毛,
            "二匹目の体": 二匹目の体,
            "二匹目の衣装": 二匹目の衣装,
            "人の性別": 人の性別,
            "人の年代": 人の年代,
            "人の髪型": 人の髪型,
            "人の髪色": 人の髪色,
            "人の服": 人の服,
            "人の体": 人の体,
            "人の表情": 人の表情,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


In [ ]:
#@title 婚活：男女の見た目 { display-mode: "form" }
#@markdown 成人の男女が会話して、最後に関係が変わる。60秒。初期値は成人の2人です。元のアカウントの人ではありません。この話を選んだときだけ使います。
#@markdown 一覧に無い見た目は、その欄に短い文を直接書く。人物は成人のみ。実在の人や元のアカウントに似せる文は書けません。

#@markdown 女性
女性の性別 = "女性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
女性の年代 = "30代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
女性の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
女性の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
女性の服 = "無地のニット" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
女性の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
女性の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 男性
男性の性別 = "男性" #@param ["女性", "男性", "その他（直接入力）"] {allow-input: true}
男性の年代 = "30代" #@param ["20代", "30代", "40代", "50代", "60代", "その他（直接入力）"] {allow-input: true}
男性の髪型 = "ショート" #@param ["ショート", "ボブ", "ロング", "結び", "その他（直接入力）"] {allow-input: true}
男性の髪色 = "黒" #@param ["黒", "暗めの茶", "明るめの茶", "グレー", "その他（直接入力）"] {allow-input: true}
男性の服 = "シャツ" #@param ["無地のニット", "シャツ", "ジャケット", "ワンピース", "その他（直接入力）"] {allow-input: true}
男性の体 = "普通" #@param ["細身", "普通", "がっしり", "背が高め", "その他（直接入力）"] {allow-input: true}
男性の表情 = "ほぼ無し" #@param ["ほぼ無し", "ナチュラル", "はっきりした眉", "やわらかい笑顔", "その他（直接入力）"] {allow-input: true}

#@markdown 場所
部屋 = "カフェ" #@param ["リビング", "キッチン", "寝室", "玄関", "カフェ", "屋外", "その他（直接入力）"] {allow-input: true}
雰囲気 = "モダン" #@param ["北欧", "和室", "モダン", "ナチュラル", "韓国風", "生活感あり", "その他（直接入力）"] {allow-input: true}
色味 = "生成り" #@param ["白と木", "グレー", "生成り", "緑がかった", "その他（直接入力）"] {allow-input: true}
光 = "暖かい室内灯" #@param ["昼の自然光", "朝", "夕方", "暖かい室内灯", "その他（直接入力）"] {allow-input: true}

#@markdown 口調
口調 = "標準語" #@param ["標準語", "関西弁", "丁寧語", "短いタメ口", "ひそひそ", "その他（直接入力）"] {allow-input: true}

import affi_bake

task = globals().get("何をする", "話でジョブを書く")
if task != "話でジョブを書く":
    print("今は見た目を使いません。「選ぶ」が「話でジョブを書く」のときだけ、このセルを実行します。")
else:
    note = affi_bake.form_status(globals().get("HANDLE"), "yako.shiawasekon")
    if note:
        print(note)
    else:
        look = affi_bake.look_from_form({
            "女性の性別": 女性の性別,
            "女性の年代": 女性の年代,
            "女性の髪型": 女性の髪型,
            "女性の髪色": 女性の髪色,
            "女性の服": 女性の服,
            "女性の体": 女性の体,
            "女性の表情": 女性の表情,
            "男性の性別": 男性の性別,
            "男性の年代": 男性の年代,
            "男性の髪型": 男性の髪型,
            "男性の髪色": 男性の髪色,
            "男性の服": 男性の服,
            "男性の体": 男性の体,
            "男性の表情": 男性の表情,
            "部屋": 部屋,
            "雰囲気": 雰囲気,
            "色味": 色味,
            "光": 光,
            "口調": 口調,
        })
        print("この話の見た目を使います。")
        print(affi_bake.STORIES[HANDLE])


# 実行

「選ぶ」のあと、このセルだけ実行します。

- 表を見る … 4ジャンルの表。ジョブは書きません
- 話でジョブを書く … 選んだ1件のジョブ。静止画が無いと止まります
- 自分の文で1本 … 手入力で I2V のコマンド。この場で焼く、を入れたときだけ mp4 を焼きます


In [ ]:
#@title 実行 { display-mode: "form" }
#@markdown 「選ぶ」のあと、このセルを実行する。表なら表。話ならジョブ。自分の文なら手入力で I2V のコマンド。
#@markdown 自分の文で「この場で焼く」を入れたときだけ、このランタイムで1本焼く。投稿しない。

import affi_bake
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

if "何をする" not in globals():
    print("先に上の「選ぶ」を実行してください。")
else:
    out = Path("/content/affi-bake") if Path("/content").is_dir() else Path("affi-bake")
    text = affi_bake.run_choice(
        何をする,
        handle=globals().get("HANDLE"),
        mode=globals().get("MODE"),
        fill_label=globals().get("中身", "テンプレ"),
        lines_text=globals().get("台詞", ""),
        image=globals().get("静止画", ""),
        look=globals().get("look"),
        prompt=globals().get("プロンプト", ""),
        duration_s=globals().get("秒", "10"),
        aspect=globals().get("画面", "9:16"),
        bake_here=bool(globals().get("この場で焼く", False)),
        data_dir=globals().get("DATA_DIR"),
        out_dir=out,
        table_out=globals().get("TABLE_OUT"),
        strong_min=int(globals().get("STRONG_MIN", 10)),
        compare_min=int(globals().get("COMPARE_MIN", 3)),
        thin_below=int(globals().get("THIN_BELOW", 10)),
        min_gap_pt=int(globals().get("MIN_GAP_PT", 10)),
    )
    if 何をする == "表を見る" and display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


# 投稿したあとの数字

今は飛ばします。

`research/affi-templates/reference-accounts/results.csv` に1行足す。空欄は0にしない。未入力のままにする。

- `hypothesis_id` … H1-1 のような番号
- `variant` … A か B
- `views` … 72時間後の再生
- `two_sec_hold_pct` … 2秒視聴のパーセント
- `diagnosis_signups` … 婚活の診断申込。無い仮説は空欄

本数が足りないと「本数不足」。基準を超えると A か B が出る。A/Bで変えるのは仮説の1項目だけ。見た目は両版で同じにする。
